# Focal Loss (RetinaNet) — Paper-to-Code Mock (Colab)

**Paper:** Focal Loss for Dense Object Detection (Lin et al., 2017) — https://arxiv.org/abs/1708.02002

Intermediate mock (~38 min). Fill in the `focal_loss_with_logits` stub, run the imbalanced-classification demo, then the sanity checks. Reference solution in the last cell.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## 1. Implement binary focal loss (YOUR TASK)

`FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)`, where `p_t` is the probability of the TRUE class.
Work from logits for numerical stability (use `binary_cross_entropy_with_logits`). `gamma=0` must reduce to plain CE.

In [ ]:
def focal_loss_with_logits(logits, targets, gamma=2.0, alpha=0.75, reduction="mean"):
    # TODO: ce = -log(p_t) per element, stably from logits (reduction='none')
    # TODO: p = sigmoid(logits); p_t = p*targets + (1-p)*(1-targets)
    # TODO: loss = (1 - p_t)**gamma * ce
    # TODO: if alpha is not None: multiply by alpha_t = alpha*targets + (1-alpha)*(1-targets)
    # TODO: apply reduction ('mean' | 'sum' | 'none') and return
    raise NotImplementedError("fill me in")

## 2. Demonstrate the benefit (imbalanced classification toy task)
~95% negatives, ~5% positives from overlapping Gaussians. Train CE vs focal; compare MINORITY-class recall.

In [ ]:
def make_data(n, frac_pos, seed):
    g = torch.Generator().manual_seed(seed)
    n_pos = int(n * frac_pos); n_neg = n - n_pos
    neg = torch.randn(n_neg, 2, generator=g)
    pos = torch.randn(n_pos, 2, generator=g) + torch.tensor([1.4, 1.4])  # overlapping
    X = torch.cat([neg, pos]); y = torch.cat([torch.zeros(n_neg), torch.ones(n_pos)])
    perm = torch.randperm(n, generator=g)
    return X[perm], y[perm]

def train(loss_kind, Xtr, ytr, seed=0, gamma=2.0, alpha=0.75, epochs=400):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 1))
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    for _ in range(epochs):
        logits = net(Xtr).squeeze(1)
        if loss_kind == "ce":
            loss = F.binary_cross_entropy_with_logits(logits, ytr)
        else:
            loss = focal_loss_with_logits(logits, ytr, gamma=gamma, alpha=alpha)
        opt.zero_grad(); loss.backward(); opt.step()
    return net

def recalls(net, X, y):
    with torch.no_grad():
        pred = (torch.sigmoid(net(X).squeeze(1)) > 0.5).float()
    rec_pos = (pred[y == 1] == 1).float().mean().item()
    rec_neg = (pred[y == 0] == 0).float().mean().item()
    return rec_pos, rec_neg

Xtr, ytr = make_data(4000, frac_pos=0.05, seed=1)
Xte, yte = make_data(4000, frac_pos=0.05, seed=2)
net_ce = train("ce",    Xtr, ytr)
net_fl = train("focal", Xtr, ytr, gamma=2.0, alpha=0.75)
ce_pos, ce_neg = recalls(net_ce, Xte, yte)
fl_pos, fl_neg = recalls(net_fl, Xte, yte)
print(f"CE    : minority recall {ce_pos:.3f}   majority recall {ce_neg:.3f}")
print(f"Focal : minority recall {fl_pos:.3f}   majority recall {fl_neg:.3f}")

## 3. Sanity checks

In [ ]:
# 1) gamma=0 reduces to plain CE
logits = torch.randn(64); targets = (torch.rand(64) > 0.5).float()
assert torch.allclose(focal_loss_with_logits(logits, targets, gamma=0.0, alpha=None),
                      F.binary_cross_entropy_with_logits(logits, targets), atol=1e-6)

# 2) easy example crushed, hard example untouched
el, et = torch.tensor([6.0]), torch.tensor([1.0]); hl, ht = torch.tensor([-6.0]), torch.tensor([1.0])
ce_e = F.binary_cross_entropy_with_logits(el, et).item(); fl_e = focal_loss_with_logits(el, et, gamma=2.0, alpha=None).item()
ce_h = F.binary_cross_entropy_with_logits(hl, ht).item(); fl_h = focal_loss_with_logits(hl, ht, gamma=2.0, alpha=None).item()
assert fl_e < ce_e * 1e-3 and fl_h > ce_h * 0.99

# 3) higher gamma down-weights an easy example more
vals = [focal_loss_with_logits(el, et, gamma=g, alpha=None).item() for g in (0.0, 1.0, 2.0, 5.0)]
assert all(vals[i+1] < vals[i] for i in range(len(vals) - 1))

# 4) reduction / shape
none_out = focal_loss_with_logits(logits, targets, reduction="none")
assert none_out.shape == logits.shape
assert torch.allclose(none_out.mean(), focal_loss_with_logits(logits, targets, reduction="mean"), atol=1e-6)
assert torch.allclose(none_out.sum(),  focal_loss_with_logits(logits, targets, reduction="sum"),  atol=1e-5)

# 5) focal beats CE on minority recall (from Part 2)
assert fl_pos > ce_pos

# 6) gradient flows
lg = torch.randn(16, requires_grad=True); tg = (torch.rand(16) > 0.5).float()
focal_loss_with_logits(lg, tg, gamma=2.0, alpha=0.75).backward()
assert lg.grad is not None and torch.isfinite(lg.grad).all() and lg.grad.abs().sum() > 0

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
def focal_loss_with_logits(logits, targets, gamma=2.0, alpha=0.75, reduction="mean"):
    ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p = torch.sigmoid(logits)
    p_t = p * targets + (1 - p) * (1 - targets)        # prob of the TRUE class
    loss = (1.0 - p_t) ** gamma * ce                    # down-weight easy examples
    if alpha is not None:
        alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
        loss = alpha_t * loss
    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    return loss